# Notebook 2 – Cross-Platform Fine-Tuning (Windows, macOS, Linux)
This notebook uses the Hugging Face ecosystem so it runs on Windows, macOS (Intel and Apple Silicon), Linux, and Google Colab.

## 1. Install Dependencies

In [ ]:
%pip install -q torch transformers datasets accelerate peft trl bitsandbytes huggingface_hub

## 2. Imports

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

## 3. Select a Model

In [ ]:
MODEL = "meta-llama/Llama-3.2-1B-Instruct"


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")


HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:

    raise ValueError(

        "HF_TOKEN not found. Check your .env.local file."

    )
print("HF_TOKEN found. Proceeding with authentication.")

## 4. Load Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL,
#     device_map="auto",
#     torch_dtype="auto"
# )

In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype="auto"
)

## 5. Load Dataset

In [ ]:
dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
dataset

## 6. Configure LoRA

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 7. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
)

## 8. Fine-Tune

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()

## 9. Save

In [ ]:
trainer.save_model("./fine_tuned_model")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


tokenizer = AutoTokenizer.from_pretrained(MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./fine_tuned_model"
)

In [ ]:
prompt = """
Write a short commercial real estate listing
for a 10,000 SF office building in Austin.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=300,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
prompt = """
Write a short commercial real estate listing
for a 10,000 SF office building in Austin.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=300,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))